# N174 · Hard图树、离线与查询综合

**目标：** 将树上路径、动态候选和状态搜索组合为算法。

**先修：** N125, N140, N141, N143, N144, N171。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** DFS入退事件；计数Trie；离线查询；状态最短路；恢复；规模约束。

## 从问题到算法

祖先最大异或查询需要的是**当前 DFS 路径**，不是整棵树。进入节点时把编号加入计数 Trie，回答该节点的全部查询，退出时删除。只建 Trie 不删除，会让兄弟分支的节点泄漏到答案中。

Trie 从高位优先选择相反位，因为任何更高位的 1 都优于所有低位之和。删除后保留的空节点不能继续参与查询，因此分支可用性取决于计数而不只是节点存在。

访问全部图节点的最短路，状态是 `(当前位置, 已访问掩码)`。同一位置在不同掩码下不是同一状态；多源 BFS 允许任选起点。树与图算法均采用迭代结构，深链不靠提高递归上限。

## 最小实现

**本章接口：** `max_genetic_difference(parents, queries)`、`shortest_path_visiting_all(graph)`

In [1]:
from collections import deque

def max_genetic_difference(parents,queries,trace=None):
    n=len(parents)
    if n==0:
        if queries:raise ValueError('Queries require a nonempty tree')
        return []
    roots=[i for i,p in enumerate(parents) if p==-1]
    if len(roots)!=1:raise ValueError('Exactly one root required')
    children=[[] for _ in range(n)]
    for node,p in enumerate(parents):
        if p!=-1:
            if not 0<=p<n or p==node:raise ValueError('Invalid parent')
            children[p].append(node)
    by_node=[[] for _ in range(n)]
    maximum=n-1
    for i,(node,value) in enumerate(queries):
        if not 0<=node<n or value<0:raise ValueError('Invalid query')
        by_node[node].append((i,value));maximum=max(maximum,value)
    width=max(1,maximum.bit_length())
    links=[[-1,-1]];counts=[0]
    def change(value,delta):
        current=0;counts[current]+=delta
        assert counts[current]>=0
        for bit in range(width-1,-1,-1):
            digit=(value>>bit)&1
            if links[current][digit]==-1:
                assert delta==1
                links[current][digit]=len(links);links.append([-1,-1]);counts.append(0)
            current=links[current][digit];counts[current]+=delta
            assert counts[current]>=0
    def query(value):
        current=0;answer=0
        assert counts[0]>0
        for bit in range(width-1,-1,-1):
            digit=(value>>bit)&1;preferred=links[current][digit^1]
            if preferred!=-1 and counts[preferred]>0:
                answer|=1<<bit;current=preferred
            else:
                current=links[current][digit]
                assert current!=-1 and counts[current]>0
        return answer
    answer=[0]*len(queries);path=[];visited=0
    stack=[(roots[0],False)]
    while stack:
        node,exiting=stack.pop()
        if exiting:
            change(node,-1);assert path.pop()==node
            continue
        visited+=1;change(node,1);path.append(node)
        for index,value in by_node[node]:answer[index]=query(value)
        if trace is not None:trace.append((node,path[:],[(v,answer[i]) for i,v in by_node[node]]))
        stack.append((node,True))
        stack.extend((child,False) for child in reversed(children[node]))
    if visited!=n:raise ValueError('Disconnected cycle in parent relation')
    assert counts[0]==0 and not path
    return answer

def shortest_path_visiting_all(graph):
    n=len(graph)
    if n<=1:return 0
    full=(1<<n)-1
    queue=deque((i,1<<i,0) for i in range(n))
    seen={(i,1<<i) for i in range(n)}
    while queue:
        node,mask,distance=queue.popleft()
        if mask==full:return distance
        for other in graph[node]:
            state=(other,mask|(1<<other))
            if state not in seen:
                seen.add(state);queue.append((*state,distance+1))
    return -1

## 正确性、前提与复杂度

DFS 进入/退出配对保证 Trie 的正计数元素集合恰是根到当前节点路径，查询因此只看合法祖先。位贪心由二进制位权支配成立。BFS 中每条状态边成本为 1，第一次到达完整掩码就是最短；按图节点去重会错误合并不同已访问信息。参照用 Floyd 距离加所有访问排列，与状态 BFS 的实现路径独立。

**代价：** 祖先查询 O((n+q)B) 时间，Trie 分配空间上界 O(nB)，B 是编号与查询值的最大位长；可选完整路径轨迹另计。全访问 BFS 至多 n·2ⁿ 个状态，时间 O((n+m)2ⁿ)，空间 O(n2ⁿ)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

trace=[]
answer=max_genetic_difference([-1,0,1,1],[[0,2],[3,2],[2,5]],trace)
show_table(['DFS node','current ancestor path','query value / xor result'],trace)
show_table(['graph','minimum walk length'],[([[1,2,3],[0],[0],[0]],shortest_path_visiting_all([[1,2,3],[0],[0],[0]]))])

DFS node,current ancestor path,query value / xor result
0,[0],"[(2, 2)]"
1,"[0, 1]",[]
2,"[0, 1, 2]","[(5, 7)]"
3,"[0, 1, 3]","[(2, 3)]"


graph,minimum walk length
"[[1, 2, 3], [0], [0], [0]]",4


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from random import Random
from itertools import permutations
assert max_genetic_difference([-1,0,1,1],[[0,2],[3,2],[2,5]])==[2,3,7]
rng=Random(174)
for n in range(1,35):
    parents=[-1]+[rng.randrange(i) for i in range(1,n)]
    queries=[(rng.randrange(n),rng.randrange(128)) for _ in range(30)]
    expected=[]
    for node,value in queries:
        candidates=[]
        while node!=-1:candidates.append(node^value);node=parents[node]
        expected.append(max(candidates))
    assert max_genetic_difference(parents,queries)==expected
assert max_genetic_difference([2,2,-1],[(0,1),(1,1)])==[3,3]
parents=[-1]+list(range(1999))
assert max_genetic_difference(parents,[(1999,2048)])[0]==max(i^2048 for i in range(2000))
def metric_reference(graph):
    n=len(graph)
    if n<=1:return 0
    dist=[[0 if i==j else float('inf') for j in range(n)] for i in range(n)]
    for i,neighbors in enumerate(graph):
        for j in neighbors:dist[i][j]=1
    for k in range(n):
        for i in range(n):
            for j in range(n):dist[i][j]=min(dist[i][j],dist[i][k]+dist[k][j])
    value=min(sum(dist[a][b] for a,b in zip(order,order[1:])) for order in permutations(range(n)))
    return -1 if value==float('inf') else value
for n in range(1,6):
    for _ in range(45):
        graph=[[] for _ in range(n)]
        for i in range(n):
            for j in range(i+1,n):
                if rng.randrange(3)==0:graph[i].append(j);graph[j].append(i)
        assert shortest_path_visiting_all(graph)==metric_reference(graph)
assert shortest_path_visiting_all([[],[]])==-1
print('N174: 所有本章断言通过')

N174: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 为树查询实现逐祖先慢参照。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较网格暴力边枚举和候选跳过。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1938. Maximum Genetic Difference Query（canonical）：[官方入口](https://leetcode.com/problems/maximum-genetic-difference-query/)
- 847. Shortest Path Visiting All Nodes（canonical）
- 2617. Minimum Number of Visited Cells in a Grid（extension）：[官方入口](https://leetcode.com/problems/minimum-number-of-visited-cells-in-a-grid/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。